# ICU Virtual Patient Feed — Patient Curation & Candidate Scoring

This notebook scans your local MIMIC-IV + MIMIC-IV-ECG download and scores every ICU stay for fit, so we can pick the 3-4 patients with the richest, most clinically interesting data for the testbed.

It scores three things:

1. **ECG density during the stay** — how many real 12-lead ECGs were recorded close to this ICU admission. We want patients with *lots* of ECGs, not just one taken at admission.
2. **Concurrent vasoactive / antiarrhythmic drug exposure** — SASI's own flagship example for Use Case B is a patient on norepinephrine, metoprolol, and amiodarone simultaneously. We're looking for real patients who match that profile.
3. **Documented hemodynamic instability while on those drugs** — real episodes of abnormal MAP/HR/SpO2 that occurred while a relevant drug was actively running. This is the real-data equivalent of SASI's "MAP 58, HR 42, SpO2 92%" example — found in actual data rather than synthesized.

**Before running:** edit the `MIMIC_ROOT` and `ECG_ROOT` paths in the config cell below to point at your local unzipped MIMIC-IV download.

This notebook does the heavy table scans in chunks (`chartevents` and `inputevents` are huge — tens of GB even unzipped), so the later cells will take a few minutes each depending on your laptop.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

## Step 0 — configure paths

Point these at your unzipped MIMIC-IV folder structure. You should have something like:

```
MIMIC_ROOT/
  hosp/   (patients.csv.gz, admissions.csv.gz, labevents.csv.gz, ...)
  icu/    (icustays.csv.gz, chartevents.csv.gz, inputevents.csv.gz, d_items.csv.gz, ...)

ECG_ROOT/
  record_list.csv
  machine_measurements.csv
  files/p..../p.........../s........./   (the actual WFDB waveform files)
```

Two thresholds matter here:

- `ECG_WINDOW_HOURS` — how far before/after the ICU stay an ECG can be and still "count." MIMIC-IV-ECG timestamps aren't all tied to an ICU visit — some are outpatient or ED-only ECGs — so we only count ones reasonably close to this admission.
- `MIN_ECG_COUNT` — the bar for "lots and lots" of ECGs. Start at 5 and raise it later if too many candidates clear it.

In [2]:
CURRENT_DIR = Path.cwd()

# If notebook is inside SASISimulator/scripts, project root is one level above
if CURRENT_DIR.name == "scripts":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"

MIMIC_ROOT = EXTERNAL_DIR / "mimic-iv-3.1"
ECG_ROOT = EXTERNAL_DIR / "mimic-iv-ecg"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MIMIC_ROOT:", MIMIC_ROOT, MIMIC_ROOT.exists())
print("ECG_ROOT:", ECG_ROOT, ECG_ROOT.exists())

PROJECT_ROOT: c:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator
MIMIC_ROOT: c:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator\data\external\mimic-iv-3.1 True
ECG_ROOT: c:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator\data\external\mimic-iv-ecg True


In [3]:
ECG_WINDOW_HOURS = 48
MIN_ECG_COUNT = 5
OUTPUT_CSV = Path("candidate_patients_scored.csv")

VASOACTIVE_DRUG_PATTERNS = [
    "norepinephrine", "noradrenaline", "epinephrine", "adrenaline",
    "phenylephrine", "vasopressin", "dopamine", "dobutamine", "milrinone",
]
ANTIARRHYTHMIC_DRUG_PATTERNS = [
    "amiodarone", "metoprolol", "esmolol", "diltiazem", "digoxin",
]
RELEVANT_DRUG_PATTERNS = VASOACTIVE_DRUG_PATTERNS + ANTIARRHYTHMIC_DRUG_PATTERNS

# Core vitals item_ids -- stable across recent MIMIC-IV versions
VITAL_ITEMS = {220045: "heart_rate", 220052: "map", 220277: "spo2"}

In [4]:
# Quick sanity check before running anything heavy
required_files = [
    MIMIC_ROOT / "icu" / "icustays.csv.gz",
    MIMIC_ROOT / "icu" / "chartevents.csv.gz",
    MIMIC_ROOT / "icu" / "inputevents.csv.gz",
    MIMIC_ROOT / "icu" / "d_items.csv.gz",
    ECG_ROOT / "record_list.csv",
]
for f in required_files:
    print(f"{'OK     ' if f.exists() else 'MISSING'}  {f}")

OK       c:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator\data\external\mimic-iv-3.1\icu\icustays.csv.gz
OK       c:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator\data\external\mimic-iv-3.1\icu\chartevents.csv.gz
OK       c:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator\data\external\mimic-iv-3.1\icu\inputevents.csv.gz
OK       c:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator\data\external\mimic-iv-3.1\icu\d_items.csv.gz
OK       c:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator\data\external\mimic-iv-ecg\record_list.csv


## Step 1 — load ICU stays and the ECG record list

`icustays.csv.gz` gives us every ICU admission with its start/end time. `record_list.csv` from MIMIC-IV-ECG gives us every 12-lead ECG ever recorded across the broader MIMIC-IV cohort — roughly 800,000 ECGs across ~160,000 patients in total, most of which have nothing to do with an ICU stay at all. The next step is what narrows that down to what we actually care about.

In [5]:
icustays = pd.read_csv(MIMIC_ROOT / "icu" / "icustays.csv.gz", parse_dates=["intime", "outtime"])
ecgs = pd.read_csv(ECG_ROOT / "record_list.csv", parse_dates=["ecg_time"])

print(f"{len(icustays):,} ICU stays")
print(f"{len(ecgs):,} ECG records across {ecgs['subject_id'].nunique():,} patients")
icustays.head(3)

94,458 ICU stays
800,035 ECG records across 161,352 patients


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535


## Step 2 — match ECGs to ICU stays (temporal join)

MIMIC-IV-ECG only gives us `subject_id` and `ecg_time` — there's no `stay_id` column linking an ECG directly to an ICU admission. So we join on `subject_id`, then keep only the ECGs whose timestamp actually falls within (or close to) the ICU stay window. This is the step that turns "this patient has an ECG somewhere in their life" into "this patient has ECGs during this specific ICU admission" — and it's also where we count how *many* ECGs fall in that window, which is the density signal we actually care about.

In [6]:
merged = icustays.merge(ecgs, on="subject_id", how="inner")

window_start = merged["intime"] - pd.Timedelta(hours=ECG_WINDOW_HOURS)
window_end = merged["outtime"] + pd.Timedelta(hours=ECG_WINDOW_HOURS)
matched = merged[(merged["ecg_time"] >= window_start) & (merged["ecg_time"] <= window_end)]

print(f"{matched['subject_id'].nunique():,} patients have >=1 ECG within the ICU stay window")
print(f"{len(matched):,} total matched ECG records")

ecg_density = (matched.groupby(["subject_id", "stay_id"])
               .agg(ecg_count_in_window=("study_id", "nunique"),
                    ecg_time_span_hours=("ecg_time", lambda s: (s.max() - s.min()).total_seconds() / 3600))
               .reset_index())

ecg_density.sort_values("ecg_count_in_window", ascending=False).head(10)

39,384 patients have >=1 ECG within the ICU stay window
141,462 total matched ECG records


,subject_id,stay_id,ecg_count_in_window,ecg_time_span_hours
32430,15910113,33409364,108,404.400000
22320,14067967,39902669,36,995.350000
50279,19206057,33473694,33,111.816667
29398,15379576,39016298,32,905.566667
35178,16418029,31193581,32,678.000000
49383,19029474,34448480,31,433.683333
13694,12502618,34357675,31,907.983333
37146,16783577,39307365,30,1141.883333
18677,13408370,30547595,28,590.850000
7320,11341217,36049628,28,795.066667


## Step 3 — find patients on vasoactive / antiarrhythmic drugs

`inputevents.csv.gz` records every continuous infusion and bolus given in the ICU — but it's one of the largest tables in MIMIC-IV, so we read it in chunks and filter to relevant `itemid`s as we go, rather than loading the whole thing into memory.

We find the relevant `itemid`s by matching drug names in `d_items.csv.gz` rather than hardcoding ID numbers — drug item IDs aren't worth memorizing exactly, and name-matching is more robust across MIMIC-IV versions anyway.

In [7]:
d_items = pd.read_csv(MIMIC_ROOT / "icu" / "d_items.csv.gz")
pattern = "|".join(RELEVANT_DRUG_PATTERNS)
relevant_items = d_items[d_items["label"].str.contains(pattern, case=False, na=False)]

print(f"Found {len(relevant_items)} matching item definitions:")
relevant_items[["itemid", "label"]]

Found 22 matching item definitions:


,itemid,label
280,221289,Epinephrine
284,221347,Amiodarone
287,221429,Esmolol
289,221468,Diltiazem
292,221653,Dobutamine
293,221662,Dopamine
299,221749,Phenylephrine
305,221906,Norepinephrine
306,221986,Milrinone
318,222315,Vasopressin


In [8]:
relevant_itemids = set(relevant_items["itemid"])
stay_ids = set(icustays["stay_id"])

chunks = []
for chunk in pd.read_csv(MIMIC_ROOT / "icu" / "inputevents.csv.gz",
                          usecols=["subject_id", "stay_id", "itemid", "starttime", "endtime",
                                   "rate", "rateuom", "amount"],
                          parse_dates=["starttime", "endtime"],
                          chunksize=2_000_000):
    chunk = chunk[chunk["stay_id"].isin(stay_ids) & chunk["itemid"].isin(relevant_itemids)]
    if len(chunk):
        chunks.append(chunk)

drugs = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
drugs = drugs.merge(relevant_items[["itemid", "label"]], on="itemid", how="left")

print(f"{len(drugs):,} relevant drug administration records across {drugs['stay_id'].nunique():,} stays")
drugs.head()

975,280 relevant drug administration records across 36,462 stays


,subject_id,stay_id,starttime,endtime,itemid,amount,rate,rateuom,label
0,10000690,37081114,2150-11-02 20:00:00,2150-11-02 22:45:00,221749,5.475664,0.600106,mcg/kg/min,Phenylephrine
1,10000690,37081114,2150-11-02 22:45:00,2150-11-02 23:36:00,221749,1.410112,0.499987,mcg/kg/min,Phenylephrine
2,10000690,37081114,2150-11-02 23:36:00,2150-11-03 00:35:00,221749,1.305181,0.400031,mcg/kg/min,Phenylephrine
3,10000690,37081114,2150-11-03 00:35:00,2150-11-03 01:07:00,221749,0.530864,0.299991,mcg/kg/min,Phenylephrine
4,10000690,37081114,2150-11-03 01:07:00,2150-11-03 02:03:00,221749,0.619409,0.200016,mcg/kg/min,Phenylephrine


## Step 4 — how many of these drugs ran *simultaneously*

A patient on norepinephrine alone is common. A patient on norepinephrine, metoprolol, *and* amiodarone all running at the same time is rare — and exactly what SASI's flagship example describes. For each stay, we sweep through the start/end times of every relevant drug and track the maximum number running concurrently at any single moment.

In [9]:
def max_concurrent(group: pd.DataFrame) -> int:
    events = []
    for _, row in group.iterrows():
        events.append((row["starttime"], 1))
        events.append((row["endtime"], -1))
    events.sort()
    running = peak = 0
    for _, delta in events:
        running += delta
        peak = max(peak, running)
    return peak

drugs_summary = (drugs.groupby("stay_id")
                  .apply(lambda g: pd.Series({
                      "max_concurrent_drugs": max_concurrent(g),
                      "n_distinct_drugs": g["label"].nunique(),
                  }))
                  .reset_index())

drugs_summary.sort_values("max_concurrent_drugs", ascending=False).head(10)

,stay_id,max_concurrent_drugs,n_distinct_drugs
17046,34669645,8,5
26299,37247308,7,7
27497,37564059,7,9
27851,37658481,7,8
33740,39268901,7,9
1237,30327685,7,9
205,30055302,7,7
28624,37876320,7,10
21105,35832565,7,4
30862,38479659,7,5


## Step 5 — real documented instability while on these drugs

`chartevents.csv.gz` is the single largest table in MIMIC-IV, so again we read in chunks and filter to just the three core vitals we need: heart rate, MAP, and SpO2.

Once we have those, we check: for every charted moment where HR, MAP, or SpO2 was abnormal, was at least one relevant drug actively running at that exact time? Those overlaps are real, documented hemodynamic derangement episodes while on a vasoactive/antiarrhythmic regimen — the real-data version of SASI's "MAP 58, HR 42, SpO2 92%" example, found rather than engineered.

In [10]:
chunks = []
for chunk in pd.read_csv(MIMIC_ROOT / "icu" / "chartevents.csv.gz",
                          usecols=["subject_id", "stay_id", "itemid", "charttime", "valuenum"],
                          parse_dates=["charttime"],
                          chunksize=5_000_000):
    chunk = chunk[chunk["stay_id"].isin(stay_ids) & chunk["itemid"].isin(VITAL_ITEMS)]
    if len(chunk):
        chunk = chunk.copy()
        chunk["vital"] = chunk["itemid"].map(VITAL_ITEMS)
        chunks.append(chunk)

vitals = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
print(f"{len(vitals):,} relevant vital sign readings across {vitals['stay_id'].nunique():,} stays")

20,416,018 relevant vital sign readings across 94,438 stays


In [11]:
def count_instability(vgrp: pd.DataFrame, dgrp: pd.DataFrame) -> int:
    if dgrp.empty:
        return 0
    hr = vgrp[vgrp["vital"] == "heart_rate"]
    map_ = vgrp[vgrp["vital"] == "map"]
    spo2 = vgrp[vgrp["vital"] == "spo2"]
    abnormal_times = pd.concat([
        hr.loc[(hr["valuenum"] < 45) | (hr["valuenum"] > 130), "charttime"],
        map_.loc[map_["valuenum"] < 65, "charttime"],
        spo2.loc[spo2["valuenum"] < 92, "charttime"],
    ])
    return sum(((dgrp["starttime"] <= t) & (dgrp["endtime"] >= t)).any() for t in abnormal_times)

results = []
for stay_id, vgrp in vitals.groupby("stay_id"):
    dgrp = drugs[drugs["stay_id"] == stay_id]
    results.append({"stay_id": stay_id, "instability_episodes": count_instability(vgrp, dgrp)})

instability = pd.DataFrame(results)
instability.sort_values("instability_episodes", ascending=False).head(10)

,stay_id,instability_episodes
33798,33576993,1079
81408,38606468,994
87955,39323481,655
22448,32380519,646
9630,31010136,566
91444,39683743,561
12306,31299423,546
53105,35642353,532
91521,39692650,528
32328,33426506,525


## Step 6 — combine everything into a single ranked candidate list

Now we merge all four signals onto every ICU stay and compute a single score. The weights below favor exactly what we're looking for: dense ECG coverage, multiple simultaneous relevant drugs, and real documented instability while on them. Longer stays get a small bonus since they generally have more usable timeline data overall, but it's a minor factor compared to the other three.

In [12]:
df = icustays.merge(ecg_density, on=["subject_id", "stay_id"], how="left")
df = df.merge(drugs_summary, on="stay_id", how="left")
df = df.merge(instability, on="stay_id", how="left")
df = df.fillna({"ecg_count_in_window": 0, "max_concurrent_drugs": 0,
                 "n_distinct_drugs": 0, "instability_episodes": 0})

df["score"] = (
    df["ecg_count_in_window"] * 4        # heavily weighted -- the "lots and lots" requirement
    + df["max_concurrent_drugs"] * 5      # 2-3 simultaneous = Use Case B fit
    + df["n_distinct_drugs"] * 2
    + df["instability_episodes"] * 6      # the actual derangement-while-on-drug signal
    + df["los"].fillna(0) * 0.5           # longer stay = more general timeline data
)
df["meets_ecg_density_bar"] = df["ecg_count_in_window"] >= MIN_ECG_COUNT

scored = df.sort_values("score", ascending=False)
scored.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {len(scored)} scored candidates to {OUTPUT_CSV.resolve()}")

cols = ["subject_id", "stay_id", "los", "ecg_count_in_window", "max_concurrent_drugs",
        "n_distinct_drugs", "instability_episodes", "meets_ecg_density_bar", "score"]
scored[cols].head(15)

Wrote 94458 scored candidates to C:\Users\dalia\OneDrive\Dokumente\Praktikum2026\SASISimulator\scripts\candidate_patients_scored.csv


,subject_id,stay_id,los,ecg_count_in_window,max_concurrent_drugs,n_distinct_drugs,instability_episodes,meets_ecg_density_bar,score
90884,19624089,33576993,91.013762,17.0,7.0,9.0,1079.0,True,6640.506881
46497,14923562,38606468,83.115046,13.0,3.0,4.0,994.0,True,6080.557523
15650,11652327,39323481,81.276134,23.0,4.0,9.0,655.0,True,4100.638067
61727,16534814,32380519,103.499005,5.0,3.0,6.0,646.0,True,3974.749502
91455,19680733,31010136,40.535347,0.0,6.0,8.0,566.0,False,3462.267674
92749,19822462,39683743,31.491956,4.0,5.0,7.0,561.0,False,3436.745978
42983,14539412,31299423,53.446678,15.0,4.0,7.0,546.0,True,3396.723339
51945,15509115,35642353,30.400463,0.0,5.0,9.0,532.0,False,3250.200231
22479,12395381,39692650,29.738507,6.0,3.0,3.0,528.0,True,3227.869253
10967,11171260,33426506,32.679630,0.0,4.0,6.0,525.0,False,3198.339815


## Next steps

Send back (or paste here) the printed top-15 table above, or the full `candidate_patients_scored.csv` if you want to look at more than 15 candidates. We'll pick the final 3-4 patients together from there, then move on to building the standardized timeline files for them.

If very few or no stays clear `MIN_ECG_COUNT`, lower it and re-run from Step 6 only (Steps 0-5 don't need to be re-run, since the underlying data hasn't changed) — or we revisit whether the 48-hour ECG matching window is too narrow for this cohort.